# 06 — Direct Preference Optimization (DPO) Training
**Goal**: Train the SFT model using Direct Preference Optimization (DPO) on execution debug trajectory pairs ($ \beta=0.1 $, LR $5\times 10^{-5}$). Produces `./checkpoints/dpo/final` for RQ5 comparison against PPO.

---

## Step 1: Environment Setup & Universal Path Resolution

In [ ]:
import sys, os, shutil

# Universal Path Resolution & Auto-Copy for Kaggle / Local / Colab
def prepare_kaggle_src():
    curr = os.path.abspath(os.getcwd())
    if os.path.exists(os.path.join(curr, 'src', 'models', 'loader.py')):
        print(f"Using local 'src' directory at {curr}")
        return curr
    
    if os.path.exists('/kaggle/input'):
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'models' in dirs and os.path.exists(os.path.join(root, 'models', 'loader.py')):
                dest = '/kaggle/working/src'
                if os.path.exists(dest):
                    shutil.rmtree(dest)
                shutil.copytree(root, dest)
                print(f"Copied 'src' from {root} to {dest}")
                return '/kaggle/working'
            elif 'src' in dirs and os.path.exists(os.path.join(root, 'src', 'models', 'loader.py')):
                src_dir = os.path.join(root, 'src')
                dest = '/kaggle/working/src'
                if os.path.exists(dest):
                    shutil.rmtree(dest)
                shutil.copytree(src_dir, dest)
                print(f"Copied 'src' from {src_dir} to {dest}")
                return '/kaggle/working'
    
    parent = os.path.abspath('..')
    if os.path.exists(os.path.join(parent, 'src')):
        return parent
    return curr

repo_root = prepare_kaggle_src()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print(f"Project path added: {repo_root}")

import torch
from datasets import load_dataset
from src.models.loader import load_model_and_tokenizer
from src.training.dpo import make_preference_pairs, run_dpo_training

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Environment initialized!")

## Step 2: Collect Preference Trajectory Pairs `(prompt, chosen, rejected)`
Runs $K=3$ debug rollouts on APPS problems. Extracts winning solutions (AC) as `chosen` and failing solutions (CE/RE/WA) as `rejected`.

In [ ]:
!pip uninstall -y torchao

MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"
SFT_CHECKPOINT = "./checkpoints/sft/final"

if not os.path.exists(SFT_CHECKPOINT) and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'adapter_config.json' in files or 'model.safetensors' in files:
            SFT_CHECKPOINT = root
            break

model_path = SFT_CHECKPOINT if os.path.exists(SFT_CHECKPOINT) else MODEL_NAME
print(f"Loading model from {model_path} for DPO pair collection in FP16 precision...")
model, tokenizer = load_model_and_tokenizer(model_name=model_path, load_in_4bit=False, lora_r=16)

print("\nLoading APPS dataset for preference pair generation via Parquet branch...")
apps = load_dataset('codeparrot/apps', revision='refs/convert/parquet', split='train[:500]')
apps_clean = apps.filter(lambda x: len(x['solutions']) > 0)

print("Generating preference dataset...")
preference_data = make_preference_pairs(apps_clean, model, tokenizer, K=3)
print(f"Total DPO preference pairs generated: {len(preference_data)}")

## Step 3: Run DPO Training
Uses PEFT reference model trick to avoid loading duplicate reference weights in VRAM.

In [ ]:
print("Starting DPO Training...")
dpo_trainer = run_dpo_training(
    model=model,
    tokenizer=tokenizer,
    preference_data=preference_data,
    output_dir="./checkpoints/dpo",
    beta=0.1,
    learning_rate=5e-5,
    num_train_epochs=3,
    per_device_train_batch_size=4,
)

print("\nDPO Training completed successfully!")
print("Saved final DPO adapter checkpoint to ./checkpoints/dpo/final")